In [1]:
# 导入必要模块
import os
import sys
import numpy as np
import cv2
import pykitti
import rospy
import rosbag
from datetime import datetime
from sensor_msgs.msg import PointCloud2, PointField, Image, Imu, NavSatFix
from std_msgs.msg import Header
from geometry_msgs.msg import Vector3
from cv_bridge import CvBridge
import struct
from sensor_msgs import point_cloud2

# 导入欧拉角转四元数依赖，缺失则提示安装
try:
    from transforms3d.euler import euler2quat
except ImportError:
    print("缺少 transforms3d 依赖，请先在终端执行安装命令：")
    print("pip install transforms3d -i https://pypi.tuna.tsinghua.edu.cn/simple")
    sys.exit(1)

In [2]:
# ---------------------- 全局参数配置（可根据自己的环境修改） ----------------------
# 1. KITTI Unsynced 数据集根目录（替换为你的实际路径）
KITTI_UNSYNCED_ROOT = "/home/jackie/Desktop/KITTI/raw_unsynced"

# 2. 数据集日期和序列编号
KITTI_DATE = "2011_10_03"
KITTI_DRIVE = "0027"

# 3. ROS Bag 保存路径和文件名
BAG_SAVE_PATH = "/home/jackie/Desktop/KITTI/raw_unsynced"
BAG_FILE_NAME = f"kitti_{KITTI_DATE}_drive_{KITTI_DRIVE}_unsynced.bag"

# 4. 传感器处理开关（True=处理，False=不处理）
PROCESS_LIDAR = True
PROCESS_OXTS = True
PROCESS_IMAGE = False  # 控制是否处理所有相机图像

# 5. ROS 话题配置（为每个传感器分配独立话题）
TOPIC_LIDAR = "/points_raw"
TOPIC_IMU = "/imu/data"
TOPIC_GPS = "/gps/fix"

# 所有相机的配置映射（相机名称 → ROS话题），自动遍历处理
CAMERA_CONFIGS = [
    ("cam0", "/cam0/image_raw"),
    ("cam1", "/cam1/image_raw"),
    ("cam2", "/cam2/image_raw"),
    ("cam3", "/cam3/image_raw")
]

# 6. 时间戳基准偏移量（一般保持 0.0 即可）
TIME_BASE = 0.0

In [3]:
# ---------------------- 工具函数：构建各类消息的 Header ----------------------
def create_lidar_header(timestamp, frame_id="velodyne"):
    """构建激光雷达消息 Header（包含时间戳和坐标系ID）"""
    header = Header()
    header.stamp = rospy.Time.from_sec(timestamp + TIME_BASE)
    header.frame_id = frame_id
    return header

def create_imu_header(timestamp, frame_id="imu"):
    """构建 IMU/GPS 消息 Header（包含时间戳和坐标系ID）"""
    header = Header()
    header.stamp = rospy.Time.from_sec(timestamp + TIME_BASE)
    header.frame_id = frame_id
    return header

def create_image_header(timestamp, frame_id="camera"):
    """构建图像消息 Header（包含时间戳和坐标系ID，可自定义相机坐标系）"""
    header = Header()
    header.stamp = rospy.Time.from_sec(timestamp + TIME_BASE)
    header.frame_id = frame_id
    return header

In [4]:
# ---------------------- 工具函数：激光雷达数据转换（.bin → PointCloud2） ----------------------
def kitti_lidar_to_ros_pointcloud2(lidar_data, timestamp):
    """
    KITTI 激光雷达数据转换为 ROS PointCloud2 消息
    参数：
        lidar_data: 单个帧的激光雷达 numpy 数组 (N, 4)
        timestamp: 该帧激光雷达的真实采集时间戳
    返回：
        pc2_msg: ROS PointCloud2 消息
    """
    # 过滤无效点云（剔除高度低于 -2.5 的点，减少噪声）
    valid_mask = (lidar_data[:, 2] > -2.5)
    lidar_data = lidar_data[valid_mask]

    # 定义 PointCloud2 消息的字段（x/y/z/intensity）
    fields = [
        PointField(name="x", offset=0, datatype=PointField.FLOAT32, count=1),
        PointField(name="y", offset=4, datatype=PointField.FLOAT32, count=1),
        PointField(name="z", offset=8, datatype=PointField.FLOAT32, count=1),
        PointField(name="intensity", offset=12, datatype=PointField.FLOAT32, count=1)
    ]

    # 构建 Header 并创建 PointCloud2 消息
    header = create_lidar_header(timestamp)
    pc2_msg = point_cloud2.create_cloud(header, fields, lidar_data)
    
    return pc2_msg

In [5]:
def euler_to_quaternion(roll, pitch, yaw):
    """欧拉角（roll/pitch/yaw）转四元数（x/y/z/w）"""
    quat = euler2quat(roll, pitch, yaw)
    return quat

def kitti_oxts_to_ros_imu(oxts_data, timestamp):
    """
    精准版IMU转换：仅线加速度(ax/ay/az)+角速度(wx/wy/wz) + 完整协方差
    索引完全匹配KITTI官方OXTs格式（从0开始）
    """
    # 核心字段：IMU原始加速度（车身FLU坐标系，索引11/12/13）
    ax = oxts_data[11]  # x轴（车辆前向）线加速度 (m/s²)
    ay = oxts_data[12]  # y轴（车辆左向）线加速度 (m/s²)
    az = oxts_data[13]  # z轴（车辆顶部）线加速度 (m/s²)
    
    # 核心字段：IMU原始角速度（索引17/18/19）
    wx = oxts_data[17]  # x轴角速率 (rad/s)
    wy = oxts_data[18]  # y轴角速率 (rad/s)
    wz = oxts_data[19]  # z轴角速率 (rad/s)
    
    # 精度参数：速度精度（用于角速度协方差，索引24）
    vel_accuracy = oxts_data[24]  # 速度精度 (m/s)

    # 初始化Imu消息并填充Header
    imu_msg = Imu()
    imu_msg.header = create_imu_header(timestamp)

    # 核心：仅填充线加速度和角速度（FLU坐标系，精准索引）
    imu_msg.linear_acceleration = Vector3(x=ax, y=ay, z=az)
    imu_msg.angular_velocity = Vector3(x=wx, y=wy, z=wz)

    # 协方差配置（按KITTI精度特性）
    # 1. 姿态协方差：无姿态数据，标注为-1（ROS标准）
    imu_msg.orientation_covariance = [-1.0, 0.0, 0.0,
                                      0.0, 0.0, 0.0,
                                      0.0, 0.0, 0.0]
    # 2. 角速度协方差：基于vel_accuracy，取平方值
    ang_cov = vel_accuracy **2  # 角速度精度与速度精度正相关
    imu_msg.angular_velocity_covariance = [ang_cov, 0.0, 0.0,
                                           0.0, ang_cov, 0.0,
                                           0.0, 0.0, ang_cov]
    # 3. 线加速度协方差：KITTI OXTs加速度精度约0.01 m/s²
    acc_cov = 0.01** 2
    imu_msg.linear_acceleration_covariance = [acc_cov, 0.0, 0.0,
                                             0.0, acc_cov, 0.0,
                                             0.0, 0.0, acc_cov]

    return imu_msg

In [6]:
def kitti_oxts_to_ros_navsatfix(oxts_data, timestamp):
    """
    精准版GPS转换：仅经纬高(lat/lon/alt) + 完整状态/精度字段
    索引完全匹配KITTI官方OXTs格式（从0开始）
    """
    # 核心字段：GPS经纬高（索引0/1/2）
    lat = oxts_data[0]  # 纬度 (deg)
    lon = oxts_data[1]  # 经度 (deg)
    alt = oxts_data[2]  # 高度 (m)
    
    # 状态/精度字段（精准索引）
    pos_accuracy = oxts_data[23]  # 位置精度（北/东向，m）→ 索引23
    navstat = oxts_data[25]       # 导航状态（0=无fix）→ 索引25
    numsats = oxts_data[26]       # 跟踪的卫星数 → 索引26
    posmode = oxts_data[27]       # GPS位置模式 → 索引27

    # 初始化NavSatFix消息并填充Header
    gps_msg = NavSatFix()
    gps_msg.header = create_imu_header(timestamp)

    # 核心：仅填充经纬高（精准索引）
    gps_msg.latitude = lat
    gps_msg.longitude = lon
    gps_msg.altitude = alt

    # 位置协方差（基于pos_accuracy）
    gps_msg.position_covariance = [pos_accuracy**2, 0.0, 0.0,
                                  0.0, pos_accuracy**2, 0.0,
                                  0.0, 0.0, (pos_accuracy*2)**2]
    gps_msg.position_covariance_type = NavSatFix.COVARIANCE_TYPE_DIAGONAL_KNOWN

    # GPS状态（按navstat/posmode判断）
    if navstat == 0 or posmode == 0:
        gps_msg.status.status = NavSatFix.STATUS_NO_FIX  # 无定位
    else:
        gps_msg.status.status = NavSatFix.STATUS_FIX      # 有定位
    gps_msg.status.service = NavSatFix.SERVICE_GPS

    return gps_msg

In [7]:
# ---------------------- 工具函数：图像数据转换（.png → Image） ----------------------
def kitti_image_to_ros_image(image_data, timestamp, camera_frame_id="camera"):
    """
    KITTI 相机图像数据转换为 ROS Image 消息
    自动判断灰度/彩色图像，选择对应编码格式
    """
    bridge = CvBridge()
    header = create_image_header(timestamp, frame_id=camera_frame_id)

    # 判断图像格式（2 维=灰度图像，3 维=彩色图像）
    if len(image_data.shape) == 2:
        # 灰度图像（KITTI cam0/cam1 通常为灰度），编码为 mono8
        image_msg = bridge.cv2_to_imgmsg(image_data, encoding="mono8")
    else:
        # 彩色图像（KITTI cam2/cam3 通常为彩色），编码为 bgr8
        image_msg = bridge.cv2_to_imgmsg(image_data, encoding="bgr8")

    # 为图像消息填充 Header
    image_msg.header = header
    
    return image_msg

In [ ]:
def convert_kitti_unsynced_to_rosbag():
    """
    核心函数：用迭代器逐帧处理 KITTI Unsynced 大数据集，避免内存溢出
    特点：
    1. 纯迭代器遍历，不转列表，低内存占用
    2. 适配各传感器帧频不一致（IMU 100Hz/雷达 10Hz/相机 10Hz）
    3. 自动处理迭代器耗尽，鲁棒性强
    """
    # ====================== 关键修复：声明全局变量 ======================
    global PROCESS_LIDAR, PROCESS_OXTS, PROCESS_IMAGE
    
    # 1. 检查数据集路径是否存在
    kitti_drive_path = os.path.join(
        KITTI_UNSYNCED_ROOT,
        KITTI_DATE,
        f"{KITTI_DATE}_drive_{KITTI_DRIVE}_sync"
    )
    if not os.path.exists(kitti_drive_path):
        print(f"❌ 错误：数据集路径不存在 → {kitti_drive_path}")
        return

    # 2. 检查并创建 Bag 保存目录
    if not os.path.exists(BAG_SAVE_PATH):
        os.makedirs(BAG_SAVE_PATH, exist_ok=True)
        print(f"📁 创建 Bag 保存目录 → {BAG_SAVE_PATH}")
    bag_full_path = os.path.join(BAG_SAVE_PATH, BAG_FILE_NAME)

    # 3. 加载 KITTI Unsynced 数据集（pykitti 生成器模式）
    print(f"📥 开始加载 KITTI 数据集 → {kitti_drive_path}")
    try:
        dataset = pykitti.raw(
            base_path=KITTI_UNSYNCED_ROOT,
            date=KITTI_DATE,
            drive=KITTI_DRIVE,
            unsynced=True,
            frames=None
        )
        # 参考时间戳数量（dataset.timestamps 是列表，可 len）
        ref_frame_count = len(dataset.timestamps) if hasattr(dataset, 'timestamps') else "未知"
        print(f"✅ 数据集加载成功，参考时间戳数量：{ref_frame_count}")
    except Exception as e:
        print(f"❌ 数据集加载失败 → {str(e)}")
        return

    # 4. 初始化 ROS Bag（写入模式）
    print(f"📝 开始创建 ROS Bag → {bag_full_path}")
    try:
        bag = rosbag.Bag(bag_full_path, 'w')
    except Exception as e:
        print(f"❌ Bag 创建失败 → {str(e)}")
        return

    # 5. 初始化 ROS 节点（仅用于时间戳处理，无需启动核心）
    rospy.init_node('kitti_unsynced_to_rosbag', anonymous=True)

    # 6. 初始化各传感器迭代器（核心：生成器 → 迭代器）
    # 激光雷达迭代器
    print(dataset.velo_files)
    print(dataset.v)
    print(dataset.data_path)

    # velo_iter = iter(dataset.velo) if hasattr(dataset, 'velo') else iter([])
    # velo_ts_iter = iter(dataset.velo_timestamps) if hasattr(dataset, 'velo_timestamps') else iter([])
    # # OXTs（IMU+GPS）迭代器
    # oxts_iter = iter(dataset.oxts) if hasattr(dataset, 'oxts') else iter([])
    # oxts_ts_iter = iter(dataset.oxts_timestamps) if hasattr(dataset, 'oxts_timestamps') else iter([])
    # # 相机迭代器字典
    # cam_iter_dict = {}
    # for cam_name, _ in CAMERA_CONFIGS:
    #     if hasattr(dataset, cam_name) and hasattr(dataset, f"{cam_name}_timestamps"):
    #         cam_iter_dict[cam_name] = (
    #             iter(getattr(dataset, cam_name)),
    #             iter(getattr(dataset, f"{cam_name}_timestamps"))
    #         )
    #     else:
    #         cam_iter_dict[cam_name] = (iter([]), iter([]))

    # # 7. 迭代处理所有帧（逐帧 next，低内存）
    # try:
    #     frame_idx = 0
    #     max_frames = 500  # 🔴 测试阶段限制500帧，正式使用改为 None
    #     process_flag = True

    #     while process_flag:
    #         # 打印进度（每 100 帧）
    #         if frame_idx % 100 == 0:
    #             print(f"⏳ 转换进度：{frame_idx} 帧")
            
    #         # 测试阶段终止条件
    #         if max_frames and frame_idx >= max_frames:
    #             print(f"📌 达到测试帧数上限（{max_frames} 帧），停止处理")
    #             break

    #         # ---------------------- 处理激光雷达 ----------------------
    #         if PROCESS_LIDAR:
    #             try:
    #                 lidar_data = next(velo_iter)
    #                 lidar_ts = next(velo_ts_iter).timestamp()
    #                 lidar_msg = kitti_lidar_to_ros_pointcloud2(lidar_data, lidar_ts)
    #                 bag.write(TOPIC_LIDAR, lidar_msg, rospy.Time.from_sec(lidar_ts + TIME_BASE))
    #             except StopIteration:
    #                 PROCESS_LIDAR = False  # 修改全局变量（已声明 global）
    #                 if frame_idx == 0:
    #                     print("⚠️ 激光雷达无数据")

    #         # ---------------------- 处理 IMU/GPS ----------------------
    #         if PROCESS_OXTS:
    #             try:
    #                 oxts_data = next(oxts_iter)
    #                 oxts_ts = next(oxts_ts_iter).timestamp()
    #                 imu_msg = kitti_oxts_to_ros_imu(oxts_data, oxts_ts)
    #                 gps_msg = kitti_oxts_to_ros_navsatfix(oxts_data, oxts_ts)
    #                 bag.write(TOPIC_IMU, imu_msg, rospy.Time.from_sec(oxts_ts + TIME_BASE))
    #                 bag.write(TOPIC_GPS, gps_msg, rospy.Time.from_sec(oxts_ts + TIME_BASE))
    #             except StopIteration:
    #                 PROCESS_OXTS = False  # 修改全局变量（已声明 global）
    #                 if frame_idx == 0:
    #                     print("⚠️ IMU/GPS 无数据")

    #         # ---------------------- 处理相机 ----------------------
    #         if PROCESS_IMAGE and len(cam_iter_dict) > 0:
    #             for cam_name, cam_topic in CAMERA_CONFIGS:
    #                 if cam_name not in cam_iter_dict:
    #                     continue
    #                 data_iter, ts_iter = cam_iter_dict[cam_name]
    #                 try:
    #                     image_data = next(data_iter)
    #                     image_ts = next(ts_iter).timestamp()
    #                     image_msg = kitti_image_to_ros_image(image_data, image_ts, camera_frame_id=cam_name)
    #                     bag.write(cam_topic, image_msg, rospy.Time.from_sec(image_ts + TIME_BASE))
    #                 except StopIteration:
    #                     cam_iter_dict.pop(cam_name)
    #                     if frame_idx == 0:
    #                         print(f"⚠️ {cam_name} 相机无数据")

    #         # ---------------------- 检查是否所有传感器处理完毕 ----------------------
    #         all_exhausted = not any([PROCESS_LIDAR, PROCESS_OXTS, (PROCESS_IMAGE and len(cam_iter_dict) > 0)])
    #         if all_exhausted:
    #             print(f"✅ 所有传感器数据处理完毕，总帧数：{frame_idx}")
    #             process_flag = False
    #             break

    #         frame_idx += 1

    #     # 8. 关闭 Bag 并提示完成
    #     bag.close()
    #     print(f"\n🎉 转换成功！Bag 文件路径：{bag_full_path}")
    #     print(f"🔧 时间戳排序命令：")
    #     print(f"rosbag filter {bag_full_path} {bag_full_path.replace('.bag', '_ordered.bag')} 'True' --sort-by=timestamp")

    # except Exception as e:
    #     print(f"❌ 转换异常 → {str(e)}")
    #     bag.close()
    #     return

In [29]:
# ---------------------- 主函数调用：触发数据转换 ----------------------
if __name__ == "__main__":
    convert_kitti_unsynced_to_rosbag()

📥 开始加载 KITTI 数据集 → /home/jackie/Desktop/KITTI/raw_unsynced/2011_10_03/2011_10_03_drive_0027_sync
✅ 数据集加载成功，参考时间戳数量：45700
📝 开始创建 ROS Bag → /home/jackie/Desktop/KITTI/raw_unsynced/kitti_2011_10_03_drive_0027_unsynced.bag
[]
/home/jackie/Desktop/KITTI/raw_unsynced/2011_10_03/2011_10_03_drive_0027_sync
